In [1]:
import requests

resposta = requests.post(
    "http://localhost:1234/v1/chat/completions",
    json={
        "messages": [
            {"role": "user", "content": "Responda em uma frase: o que é churn de clientes?"}
        ],
        "temperature": 0.7
    }
)

print(resposta.json())

{'id': 'chatcmpl-ux5bwryfetdmu6srdfx2nh', 'object': 'chat.completion', 'created': 1783134635, 'model': 'meta-llama-3-8b-instruct', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'O "churn" (também conhecido como "turnover") de clientes é a taxa de mudança ou perda de clientes em um período determinado, seja por cancelamento da assinatura, falta de pagamento ou outra razão.', 'reasoning_content': '', 'tool_calls': []}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 24, 'completion_tokens': 53, 'total_tokens': 77, 'completion_tokens_details': {'reasoning_tokens': 0}}, 'stats': {}, 'system_fingerprint': 'meta-llama-3-8b-instruct'}


In [3]:
texto = resposta.json()["choices"][0]["message"]["content"]
print(texto)

O "churn" (também conhecido como "turnover") de clientes é a taxa de mudança ou perda de clientes em um período determinado, seja por cancelamento da assinatura, falta de pagamento ou outra razão.


In [4]:
import joblib

modelo = joblib.load("modelo_churn.pkl")
colunas_modelo = joblib.load("colunas_modelo.pkl")

In [5]:
import pandas as pd

# um cliente fictício de exemplo 
cliente_exemplo = {
    "tenure": 2,
    "MonthlyCharges": 95.0,
    "TotalCharges": 190.0,
    "Contract_One year": 0,
    "Contract_Two year": 0,
}

# monta um DataFrame com todas as colunas que o modelo espera, preenchendo 0 onde faltar
X_novo = pd.DataFrame([cliente_exemplo])
X_novo = X_novo.reindex(columns=colunas_modelo, fill_value=0)

probabilidade = modelo.predict_proba(X_novo)[:, 1][0]
print(f"Probabilidade de churn: {probabilidade:.2%}")

Probabilidade de churn: 42.50%


In [6]:
prob_formatada = f"{probabilidade:.1%}"

prompt = f"""Você é um assistente de retenção de clientes. Um modelo de Machine Learning
calculou que um cliente tem {prob_formatada} de probabilidade de churn (cancelamento).

Dados do cliente:
- Tempo de contrato (tenure): {cliente_exemplo['tenure']} meses
- Cobrança mensal: R${cliente_exemplo['MonthlyCharges']}
- Total gasto até agora: R${cliente_exemplo['TotalCharges']}

O modelo identificou que, em geral, os fatores mais importantes para prever churn são:
tempo de contrato baixo, cobrança mensal alta, e ausência de contrato de longo prazo.

Com base APENAS nesses dados, escreva:
1. Uma explicação curta (2-3 frases) de por que esse cliente pode estar em risco.
2. Uma sugestão de ação de retenção específica para esse caso.

Não invente informações que não foram fornecidas."""

resposta = requests.post(
    "http://localhost:1234/v1/chat/completions",
    json={
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.3
    }
)

explicacao = resposta.json()["choices"][0]["message"]["content"]
print(explicacao)

**1. Explicação curta do risco de churn:**
Este cliente tem um tempo de contrato relativamente baixo (2 meses), o que pode indicar uma falta de compromisso com a empresa. Além disso, a cobrança mensal é alta (R$95.0) e não há informações sobre um contrato de longo prazo, o que podem ser fatores contribuintes para o risco de cancelamento.

**2. Sugestão de ação de retenção:**
Considerando os dados fornecidos, uma sugestão de ação de retenção pode ser oferecer ao cliente um plano de pagamento mais flexível ou uma promoção especial para contratos de longo prazo, visando demonstrar a empresa's compromisso com sua satisfação e reduzir o risco de cancelamento.


In [13]:
def prever_churn(dados_cliente: dict) -> dict:
    X_novo = pd.DataFrame([dados_cliente])
    X_novo = X_novo.reindex(columns=colunas_modelo, fill_value=0)

    probabilidade = modelo.predict_proba(X_novo)[:, 1][0]
    prob_formatada = f"{probabilidade:.1%}"

    # decide o enquadramento ANTES de mandar pro LLM
    if probabilidade >= 0.5:
        nivel_risco = "ALTO risco de cancelamento"
        instrucao = "Explique quais fatores aumentam o risco deste cliente e sugira uma ação de retenção urgente."
    elif probabilidade >= 0.3:
        nivel_risco = "risco MODERADO de cancelamento"
        instrucao = "Explique os fatores de atenção e sugira uma ação preventiva."
    else:
        nivel_risco = "BAIXO risco de cancelamento"
        instrucao = "Explique por que este cliente parece estável e sugira uma ação de manutenção do relacionamento (não de retenção urgente)."

    prompt = f""" Responda sempre em Potugues do Brasil. Você é um assistente de retenção de clientes. Um modelo de Machine Learning
calculou que um cliente tem {prob_formatada} de probabilidade de churn — classificado como {nivel_risco}.

Dados do cliente:
- Tempo de contrato (tenure): {dados_cliente.get('tenure', 'não informado')} meses
- Cobrança mensal: R${dados_cliente.get('MonthlyCharges', 'não informado')}
- Total gasto até agora: R${dados_cliente.get('TotalCharges', 'não informado')}

{instrucao}

Não invente informações que não foram fornecidas. Seja coerente com o nível de risco informado."""

    resposta = requests.post(
        "http://localhost:1234/v1/chat/completions",
        json={
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.3
        }
    )
    explicacao = resposta.json()["choices"][0]["message"]["content"]

    return {
        "probabilidade_churn": round(float(probabilidade), 4),
        "nivel_risco": nivel_risco,
        "explicacao": explicacao
    }

In [14]:
resultado = prever_churn({
    "tenure": 60,
    "MonthlyCharges": 45.0,
    "TotalCharges": 2700.0,
    "Contract_Two year": 1
})

print(resultado)

{'probabilidade_churn': 0.07, 'nivel_risco': 'BAIXO risco de cancelamento', 'explicacao': 'Olá, cliente!\n\nAqui na nossa equipe de atendimento ao cliente, recebemos um relatório do nosso modelo de Machine Learning indicando que você tem 7,0% de probabilidade de cancelamento, classificado como BAIXO risco de cancelamento.\n\nCom base nos dados fornecidos, podemos ver que você tem um tempo de contrato (tenure) de 60 meses, o que é uma indicação positiva da estabilidade do relacionamento. Além disso, sua cobrança mensal é de R$45,0 e seu total gasto até agora é de R$2700,00, o que demonstra um compromisso consistente com a nossa empresa.\n\nCom esses dados, podemos concluir que você parece estar estável em relação ao nosso serviço. No entanto, para manter o relacionamento saudável e evitar qualquer problema futuro, gostaríamos de sugerir uma ação de manutenção:\n\n* Oferecemos um check-up regular do seu plano para garantir que esteja satisfeito com as nossas ofertas e serviços.\n* Vamos 